In [ ]:
pip install pandas tqdm

In [5]:
import pandas as pd
import requests
import zipfile
import io

# ── Columns to load from raw files ──
cols_needed = [
    'as_of_year',
    'state_code',
    'msamd',
    'msamd_name',
    'action_taken_name',
    'loan_type_name',
    'loan_purpose_name',
    'lien_status_name',
]

# ── Filter values ──
keep_purpose = ['Home purchase', 'Refinancing']
keep_action  = [
    'Loan originated',
    'Application denied by financial institution'
]
keep_lien    = ['Secured by a first lien']

# ── Years to process ──
years = list(range(2007, 2018))

# ── URL pattern ──
# Using "all-records" so we get both originated AND denied
def get_url(year):
    return (
        f"https://files.consumerfinance.gov/hmda-historic-loan-data/"
        f"hmda_{year}_nationwide_all-records_labels.zip"
    )

all_agg = []  # stores only tiny aggregated tables, not raw rows

for year in years:
    print(f"Processing {year}...")

    url      = get_url(year)
    response = requests.get(url, stream=True, timeout=300)
    z        = zipfile.ZipFile(io.BytesIO(response.content))
    fname    = z.namelist()[0]

    # Read in chunks — never loads full file into RAM
    chunks = pd.read_csv(
        z.open(fname),
        usecols=cols_needed,
        dtype=str,
        low_memory=False,
        chunksize=100_000
    )

    # Filter each chunk as it is read, then combine
    year_df = pd.concat([
        chunk[
            chunk['lien_status_name'].isin(keep_lien) &
            chunk['loan_purpose_name'].isin(keep_purpose) &
            chunk['action_taken_name'].isin(keep_action)
        ]
        for chunk in chunks
    ])

    # Add binary flag columns for aggregation
    year_df['is_originated']   = (year_df['action_taken_name'] == 'Loan originated').astype(int)
    year_df['is_denied']       = (year_df['action_taken_name'] == 'Application denied by financial institution').astype(int)
    year_df['is_conventional'] = (year_df['loan_type_name'] == 'Conventional').astype(int)
    year_df['is_govt_backed']  = year_df['loan_type_name'].isin(['FHA-insured', 'VA-guaranteed', 'FSA/RHS']).astype(int)
    year_df['is_purchase']     = (year_df['loan_purpose_name'] == 'Home purchase').astype(int)

    # Aggregate immediately — discard raw rows after this
    agg = (
        year_df
        .groupby(['as_of_year', 'state_code', 'msamd', 'msamd_name'])
        .agg(
            total_applications = ('action_taken_name',  'count'),
            total_originated   = ('is_originated',      'sum'),
            total_denied       = ('is_denied',          'sum'),
            total_conventional = ('is_conventional',    'sum'),
            total_govt_backed  = ('is_govt_backed',     'sum'),
            total_purchase     = ('is_purchase',        'sum'),
        )
        .reset_index()
    )

    # Compute rate columns
    agg['denial_rate']        = (agg['total_denied']       / agg['total_applications']).round(4)
    agg['origination_rate']   = (agg['total_originated']   / agg['total_applications']).round(4)
    agg['conventional_share'] = (agg['total_conventional'] / agg['total_applications']).round(4)
    agg['govt_backed_share']  = (agg['total_govt_backed']  / agg['total_applications']).round(4)
    agg['purchase_share']     = (agg['total_purchase']     / agg['total_applications']).round(4)

    # Keep only the final 12 columns — drop intermediate counts
    agg = agg[[
        'as_of_year',
        'state_code',
        'msamd',
        'msamd_name',
        'total_applications',
        'total_originated',
        'total_denied',
        'denial_rate',
        'origination_rate',
        'conventional_share',
        'govt_backed_share',
        'purchase_share',
    ]]

    all_agg.append(agg)

    # Free memory before next year
    del year_df, chunks, z, response
    print(f"  ✓ {year} done — {len(agg):,} aggregated rows")

# Combine all years — tiny since each year is already aggregated
final = pd.concat(all_agg, ignore_index=True)
final.to_csv('combined_clean_data.csv', index=False)
print(f"\nDone. Final shape: {final.shape}")
print(f"Years: {sorted(final['as_of_year'].unique())}")
print(f"States: {final['state_code'].nunique()}")
print(f"MSAs: {final['msamd'].nunique()}")
print(final.head())

Processing 2007...
  ✓ 2007 done — 440 aggregated rows
Processing 2008...
  ✓ 2008 done — 440 aggregated rows
Processing 2009...
  ✓ 2009 done — 444 aggregated rows
Processing 2010...
  ✓ 2010 done — 444 aggregated rows
Processing 2011...
  ✓ 2011 done — 444 aggregated rows
Processing 2012...
  ✓ 2012 done — 444 aggregated rows
Processing 2013...
  ✓ 2013 done — 444 aggregated rows
Processing 2014...
  ✓ 2014 done — 462 aggregated rows
Processing 2015...
  ✓ 2015 done — 462 aggregated rows
Processing 2016...
  ✓ 2016 done — 463 aggregated rows
Processing 2017...
  ✓ 2017 done — 463 aggregated rows

Done. Final shape: (4950, 12)
Years: ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017']
States: 52
MSAs: 434
  as_of_year state_code  msamd               msamd_name  total_applications  \
0       2007          1  11500    Anniston, Oxford - AL                4202   
1       2007          1  12220     Auburn, Opelika - AL                5777   
2       20

In [ ]:
######################
import pandas as pd
import requests
import zipfile
import io

cols_needed = [
    'as_of_year',
    'action_taken_name',
    'loan_purpose_name',
    'loan_type_name',
    'state_abbr',
]

keep_purpose = ['Home purchase', 'Refinancing']
keep_action  = ['Loan originated',
                'Application denied by financial institution',
                'Application approved but not accepted']

years = [2007, 2008, 2009, 2010, 2011, 2013, 2014, 2015, 2016, 2017]
all_agg = []  # only storing tiny aggregated tables, not raw data

for year in years:
    print(f"Processing {year}...")

    url = f"https://files.consumerfinance.gov/hmda-historic-loan-data/hmda_{year}_nationwide_first-lien-owner-occupied-1-4-family-records_labels.zip"

    response = requests.get(url, stream=True, timeout=300)
    z = zipfile.ZipFile(io.BytesIO(response.content))
    fname = z.namelist()[0]

    # read in chunks — never loads full file into RAM
    chunks = pd.read_csv(
        z.open(fname),
        usecols=cols_needed,
        dtype=str,
        low_memory=False,
        chunksize=100_000   # process 100k rows at a time
    )

    year_df = pd.concat([
        chunk[
            chunk['loan_purpose_name'].isin(keep_purpose) &
            chunk['action_taken_name'].isin(keep_action)
        ]
        for chunk in chunks
    ])

    # aggregate immediately — discard raw rows
    agg = year_df.groupby([
        'as_of_year', 'state_abbr',
        'loan_purpose_name', 'loan_type_name',
        'action_taken_name'
    ]).size().reset_index(name='count')

    all_agg.append(agg)

    # explicitly free memory
    del year_df, chunks, z, response

    print(f"  ✓ {year} done — {len(agg)} aggregated rows")

# now combine — this is tiny, won't crash
final = pd.concat(all_agg, ignore_index=True)
final.to_csv('hmda_aggregated.csv', index=False)
print(f"Done. Final shape: {final.shape}")